In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [6]:
ROOT = Path(r"C:\Users\hoo58\Anti-Churn-Committee\Anti-Churn-Committee")

DATA_PROCESSED_DIR = ROOT / "DATA"

USER_PROFILE_FILE = "User_Profile.csv"
EVENT_LOG_FILE = "Event_Log.csv"

USER_PROFILE_PATH = DATA_PROCESSED_DIR / USER_PROFILE_FILE
EVENT_LOG_PATH = DATA_PROCESSED_DIR / EVENT_LOG_FILE

print("USER_PROFILE_PATH:", USER_PROFILE_PATH)
print("EVENT_LOG_PATH   :", EVENT_LOG_PATH)


USER_PROFILE_PATH: C:\Users\hoo58\Anti-Churn-Committee\Anti-Churn-Committee\DATA\User_Profile.csv
EVENT_LOG_PATH   : C:\Users\hoo58\Anti-Churn-Committee\Anti-Churn-Committee\DATA\Event_Log.csv


In [7]:
df_raw01 = pd.read_csv(USER_PROFILE_PATH)
df_raw02 = pd.read_csv(EVENT_LOG_PATH)

print("df_raw01 shape:", df_raw01.shape)
print("df_raw02 shape:", df_raw02.shape)

display(df_raw01.head())
display(df_raw02.head())


user_profile = df_raw01.copy()
event_log = df_raw02.copy()

df_raw01 shape: (12500, 6)
df_raw02 shape: (1757262, 5)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


In [8]:
user_profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   User_ID      12500 non-null  object
 1   가입일자         12500 non-null  object
 2   가입경로         12363 non-null  object
 3   기기           12379 non-null  object
 4   알림수신동의여부     12384 non-null  object
 5   알림수신동의_변경일자  1976 non-null   object
dtypes: object(6)
memory usage: 586.1+ KB


In [9]:
event_log.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1757262 entries, 0 to 1757261
Data columns (total 5 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   User_ID     object
 1   Event_Time  object
 2   Event_Type  object
 3   Session_ID  object
 4   알림_유형       object
dtypes: object(5)
memory usage: 67.0+ MB


In [10]:
# ============================================================
# 1. 기본 전처리
# ============================================================

user_profile = df_raw01.copy()
event_log = df_raw02.copy()

# 컬럼명 공백 제거
user_profile.columns = user_profile.columns.str.strip()
event_log.columns = event_log.columns.str.strip()

# 문자열 값 공백 제거
user_text_cols = user_profile.select_dtypes(include=["object", "string"]).columns
event_text_cols = event_log.select_dtypes(include=["object", "string"]).columns

user_profile[user_text_cols] = user_profile[user_text_cols].apply(lambda col: col.astype("string").str.strip())
event_log[event_text_cols] = event_log[event_text_cols].apply(lambda col: col.astype("string").str.strip())

# User_ID 타입 통일
user_profile["User_ID"] = user_profile["User_ID"].astype("string")
event_log["User_ID"] = event_log["User_ID"].astype("string")

# 날짜 변환
user_profile["가입일자"] = pd.to_datetime(user_profile["가입일자"], errors="coerce")
user_profile["알림수신동의_변경일자"] = pd.to_datetime(user_profile["알림수신동의_변경일자"], errors="coerce")
event_log["Event_Time"] = pd.to_datetime(event_log["Event_Time"], errors="coerce")

# 분석에 자주 쓸 날짜/월 컬럼
user_profile["가입일"] = user_profile["가입일자"].dt.normalize()
user_profile["가입월"] = user_profile["가입일자"].dt.to_period("M").astype("string")

event_log["Event_Date"] = event_log["Event_Time"].dt.normalize()
event_log["Event_Month"] = event_log["Event_Time"].dt.to_period("M").astype("string")
event_log["Event_Hour"] = event_log["Event_Time"].dt.hour

print("user_profile shape:", user_profile.shape)
print("event_log shape   :", event_log.shape)

user_profile shape: (12500, 8)
event_log shape   : (1757262, 8)


In [11]:
# 일/월
user_profile['가입월'] = user_profile['가입일자'].dt.month

In [12]:
user_profile.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입일,가입월
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25,1
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24,2025-05-06,5
2,U0000003,2025-05-14,오가닉,iOS,False,NaT,2025-05-14,5
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT,2025-02-23,2
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT,2025-02-18,2


In [13]:
# ============================================================
# 2. 요일 파생컬럼 추가
# ============================================================

# 요일 매핑
weekday_map = {
    0: "월",
    1: "화",
    2: "수",
    3: "목",
    4: "금",
    5: "토",
    6: "일"
}

# 가입일 기준 요일
user_profile["가입요일번호"] = user_profile["가입일자"].dt.dayofweek
user_profile["가입요일"] = user_profile["가입요일번호"].map(weekday_map)

# 이벤트 발생일 기준 요일
event_log["Event_요일번호"] = event_log["Event_Time"].dt.dayofweek
event_log["Event_요일"] = event_log["Event_요일번호"].map(weekday_map)

display(user_profile[["User_ID", "가입일자", "가입요일번호", "가입요일"]].head())
display(event_log[["User_ID", "Event_Time", "Event_요일번호", "Event_요일"]].head())

,User_ID,가입일자,가입요일번호,가입요일
0,U0000001,2025-01-25,5,토
1,U0000002,2025-05-06,1,화
2,U0000003,2025-05-14,2,수
3,U0000004,2025-02-23,6,일
4,U0000005,2025-02-18,1,화


,User_ID,Event_Time,Event_요일번호,Event_요일
0,U0000001,2025-01-25 07:25:45,5,토
1,U0000001,2025-01-25 07:26:15,5,토
2,U0000001,2025-01-25 07:26:55,5,토
3,U0000001,2025-01-25 07:27:55,5,토
4,U0000001,2025-01-25 20:30:00,5,토


In [14]:
# 3월 10~14일 로그 장애 구간 플래그 마킹
event_log['is_log_anomaly'] = event_log['Event_Time'].between(
    pd.Timestamp('2025-03-10'),
    pd.Timestamp('2025-03-14 23:59:59')
)

anomaly_count = event_log['is_log_anomaly'].sum()
total         = len(event_log)
print(f"로그 장애 구간 이벤트 수: {anomaly_count:,}건 ({anomaly_count / total * 100:.1f}%)")
print(f"정상 구간 이벤트 수:      {total - anomaly_count:,}건 ({(total - anomaly_count) / total * 100:.1f}%)")

로그 장애 구간 이벤트 수: 20,862건 (1.2%)
정상 구간 이벤트 수:      1,736,400건 (98.8%)


In [15]:
# 유저 정합성 체크
profile_ids    = set(event_log['User_ID'])
event_ids      = set(event_log['User_ID'])
inactive_users = profile_ids - event_ids

print(f"프로필 등록 유저: {len(profile_ids):,}명")
print(f"이벤트 발생 유저: {len(event_ids):,}명")
print(f"완전 미활동 유저: {len(inactive_users)}명  ← 이벤트 로그에 없음")

# 미활동 유저 프로필 살펴보기
inactive_df = user_profile[user_profile['User_ID'].isin(inactive_users)].copy()
print(f"\n미활동 유저 가입경로 분포:\n{inactive_df['가입경로'].value_counts()}")
print(f"\n미활동 유저 가입월 분포:\n{inactive_df['가입일자'].dt.to_period('M').value_counts().sort_index()}")

프로필 등록 유저: 12,453명
이벤트 발생 유저: 12,453명
완전 미활동 유저: 0명  ← 이벤트 로그에 없음

미활동 유저 가입경로 분포:
Series([], Name: count, dtype: Int64)

미활동 유저 가입월 분포:
Series([], Freq: M, Name: count, dtype: int64)


In [16]:
event_log.head()

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,Event_Date,Event_Month,Event_Hour,Event_요일번호,Event_요일,is_log_anomaly
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,<NA>,2025-01-25,2025-01,7,5,토,False
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,<NA>,2025-01-25,2025-01,7,5,토,False
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,<NA>,2025-01-25,2025-01,7,5,토,False
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,<NA>,2025-01-25,2025-01,7,5,토,False
4,U0000001,2025-01-25 20:30:00,알림수신,<NA>,광고성,2025-01-25,2025-01,20,5,토,False


In [17]:
user_profile.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입일,가입월,가입요일번호,가입요일
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25,1,5,토
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24,2025-05-06,5,1,화
2,U0000003,2025-05-14,오가닉,iOS,False,NaT,2025-05-14,5,2,수
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT,2025-02-23,2,6,일
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT,2025-02-18,2,1,화


In [18]:
# 유저별 첫 이벤트 시간
first_event = event_log.groupby('User_ID')['Event_Time'].min().reset_index()
first_event.columns = ['User_ID', '첫이벤트_시간']

# 가입일시 붙이기
first_event = first_event.merge(user_profile[['User_ID', '가입일자']], on='User_ID', how='left')

# 시간 차이 계산
first_event['첫행동_소요시간_시간'] = (
    first_event['첫이벤트_시간'] - first_event['가입일자']
).dt.total_seconds() / 3600

first_event = first_event[first_event['첫행동_소요시간_시간'] >= 0]

print("[ 첫 행동까지 소요시간 ]")
display(first_event['첫행동_소요시간_시간'].describe())

[ 첫 행동까지 소요시간 ]


count    12453.000000
mean        11.182576
std          6.901402
min          6.945556
25%          8.216667
50%          9.671111
75%         12.183333
max        135.587222
Name: 첫행동_소요시간_시간, dtype: float64

In [19]:
# 알림 이벤트는
event_no_alarm = event_log[
    ~event_log['Event_Type'].isin(['알림수신', '알림오픈'])
].copy()

# 세션별 이벤트 수
session_event_count = event_no_alarm.groupby(
    ['User_ID', 'Session_ID']
).size().reset_index(name='세션내_이벤트수')

# 유저별 세션당 평균 이벤트 수
user_session_avg = session_event_count.groupby('User_ID')['세션내_이벤트수'].mean().reset_index()
user_session_avg.columns = ['User_ID', '세션당_평균이벤트수']
user_session_avg['세션당_평균이벤트수'] = user_session_avg['세션당_평균이벤트수'].round(2)

print("[ 세션당 평균 이벤트 수 기술통계 ]")
display(user_session_avg['세션당_평균이벤트수'].describe())

display(user_session_avg.head())

[ 세션당 평균 이벤트 수 기술통계 ]


count    12451.000000
mean         2.124248
std          0.516579
min          1.000000
25%          1.810000
50%          2.000000
75%          2.330000
max          6.000000
Name: 세션당_평균이벤트수, dtype: float64

,User_ID,세션당_평균이벤트수
0,U0000001,2.07
1,U0000002,2.62
2,U0000003,1.50
3,U0000004,1.79
4,U0000005,1.85


In [21]:
# 전처리 완료된 df를 CSV로 저장
user_profile.to_csv('../../data/user_profile_preprocessed.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료: user_profile_preprocessed.csv ({len(user_profile):,}행 × {user_profile.shape[1]}컬럼)")

저장 완료: user_profile_preprocessed.csv (12,500행 × 10컬럼)


In [22]:
event_log.to_csv('../../data/event_log_preprocessed.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료: event_log_preprocessed.csv ({len(event_log):,}행 × {event_log.shape[1]}컬럼)")

저장 완료: event_log_preprocessed.csv (1,757,262행 × 11컬럼)
